# Tarea 3: clasificación de mensajes

Voy a comparar dos modelos para separar mensajes normales y spam

## datos

Voy a usar [SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) de UCI, los mensajes ya vienen marcados como `ham` o `spam`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import urllib.request
from io import BytesIO
from zipfile import ZipFile

semilla = 42

In [ ]:
url_datos = 'https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'

# lo leo directo del zip para no pedir una carga manual
respuesta = urllib.request.urlopen(url_datos)
archivo = ZipFile(BytesIO(respuesta.read()))
datos = pd.read_csv(
    archivo.open('SMSSpamCollection'), sep='\t', header=None,
    names=['etiqueta', 'texto']
)
archivo.close()

datos.head()

## una revisión rápida

In [ ]:
datos.isna().sum().sum(), datos.duplicated(subset='texto').sum(), datos['etiqueta'].value_counts()

In [ ]:
# quito mensajes repetidos para que uno no caiga en entrenamiento y otro en prueba
datos = datos.drop_duplicates(subset='texto').reset_index(drop=True)

datos['cantidad_palabras'] = datos['texto'].str.split().str.len()
datos['cantidad_caracteres'] = datos['texto'].str.len()
datos['objetivo'] = datos['etiqueta'].map({'ham': 0, 'spam': 1})

resumen_clases = datos.groupby('etiqueta')[['cantidad_palabras', 'cantidad_caracteres']].mean().round(1)
resumen_clases

In [ ]:
conteo_clases = datos['etiqueta'].value_counts()
conteo_clases.plot(kind='bar', color=['steelblue', 'orange'])
plt.title('mensajes por clase')
plt.show()

In [ ]:
datos.hist(column='cantidad_palabras', by='etiqueta', bins=25, figsize=(8, 3))
plt.show()

Hay mucho menos spam, por eso voy a fijarme en F1, precision y recall y no solo en accuracy

## limpieza

Solo voy a pasar a minúsculas y quitar algunos símbolos, los enlaces y números los dejo como palabras

In [ ]:
datos['texto_limpio'] = (
    datos['texto'].str.lower()
    .str.replace(r'https?://\S+|www\.\S+', ' url ', regex=True)
    .str.replace(r'\d+', ' numero ', regex=True)
    .str.replace(r'[^a-z\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

datos[['etiqueta', 'texto', 'texto_limpio']].sample(5, random_state=semilla)

## separar los datos

Dejo 20% para probar al final y uso `stratify` para mantener la cantidad de spam

In [ ]:
from sklearn.model_selection import train_test_split

texto_entreno, texto_prueba, objetivo_entreno, objetivo_prueba = train_test_split(
    datos['texto_limpio'], datos['objetivo'], test_size=.20,
    random_state=semilla, stratify=datos['objetivo']
)

len(texto_entreno), len(texto_prueba)

## experimento

Uso TF-IDF y validación de 5 partes para los dos modelos

- n-gramas `(1,1)` y `(1,2)`
- frecuencia mínima 1 y 2
- regresión logística con `c` de 0.5, 1 y 2, normal o balanceada
- naive bayes con `alpha` de 0.1, 0.5, 1 y 2

Voy a escoger el resultado con mejor F1 para spam

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

In [ ]:
validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)

modelo_logistico = Pipeline([
    ('vectorizador', TfidfVectorizer()),
    ('clasificador', LogisticRegression(max_iter=1000, random_state=semilla))
])
parametros_logistico = {
    'vectorizador__ngram_range': [(1, 1), (1, 2)],
    'vectorizador__min_df': [1, 2],
    'clasificador__C': [.5, 1, 2],
    'clasificador__class_weight': [None, 'balanced']
}

modelo_bayes = Pipeline([
    ('vectorizador', TfidfVectorizer()),
    ('clasificador', MultinomialNB())
])
parametros_bayes = {
    'vectorizador__ngram_range': [(1, 1), (1, 2)],
    'vectorizador__min_df': [1, 2],
    'clasificador__alpha': [.1, .5, 1, 2]
}

In [ ]:
# primero hago la regresión logística
busqueda_logistica = GridSearchCV(
    modelo_logistico, parametros_logistico,
    scoring=['accuracy', 'precision', 'recall', 'f1'], refit='f1',
    cv=validacion, n_jobs=-1
)
busqueda_logistica.fit(texto_entreno, objetivo_entreno)

# ahora repito con naive bayes
busqueda_bayes = GridSearchCV(
    modelo_bayes, parametros_bayes,
    scoring=['accuracy', 'precision', 'recall', 'f1'], refit='f1',
    cv=validacion, n_jobs=-1
)
busqueda_bayes.fit(texto_entreno, objetivo_entreno)

### resultados de la validación

Aquí veo cuál fue la mejor combinación de cada modelo

In [ ]:
busqueda_logistica.best_params_, round(busqueda_logistica.best_score_, 4), busqueda_bayes.best_params_, round(busqueda_bayes.best_score_, 4)

In [ ]:
pos_logistica = busqueda_logistica.best_index_
pos_bayes = busqueda_bayes.best_index_

tabla_cv = pd.DataFrame([
    ['regresion logistica',
     busqueda_logistica.cv_results_['mean_test_accuracy'][pos_logistica],
     busqueda_logistica.cv_results_['mean_test_precision'][pos_logistica],
     busqueda_logistica.cv_results_['mean_test_recall'][pos_logistica],
     busqueda_logistica.cv_results_['mean_test_f1'][pos_logistica]],
    ['naive bayes',
     busqueda_bayes.cv_results_['mean_test_accuracy'][pos_bayes],
     busqueda_bayes.cv_results_['mean_test_precision'][pos_bayes],
     busqueda_bayes.cv_results_['mean_test_recall'][pos_bayes],
     busqueda_bayes.cv_results_['mean_test_f1'][pos_bayes]]
], columns=['modelo', 'accuracy', 'precision', 'recall', 'f1'])

tabla_cv.round(4)

## prueba final

Ahora pruebo la mejor versión de cada modelo con los mensajes que había separado

In [ ]:
prediccion_logistica = busqueda_logistica.predict(texto_prueba)
prediccion_bayes = busqueda_bayes.predict(texto_prueba)

accuracy_logistica = accuracy_score(objetivo_prueba, prediccion_logistica)
precision_logistica = precision_score(objetivo_prueba, prediccion_logistica)
recall_logistica = recall_score(objetivo_prueba, prediccion_logistica)
f1_logistica = f1_score(objetivo_prueba, prediccion_logistica)

accuracy_bayes = accuracy_score(objetivo_prueba, prediccion_bayes)
precision_bayes = precision_score(objetivo_prueba, prediccion_bayes)
recall_bayes = recall_score(objetivo_prueba, prediccion_bayes)
f1_bayes = f1_score(objetivo_prueba, prediccion_bayes)

tabla_prueba = pd.DataFrame([
    ['regresion logistica', accuracy_logistica, precision_logistica, recall_logistica, f1_logistica],
    ['naive bayes', accuracy_bayes, precision_bayes, recall_bayes, f1_bayes]
], columns=['modelo', 'accuracy', 'precision', 'recall', 'f1'])

tabla_prueba.round(4)

In [ ]:
# aquí veo los aciertos y errores de cada uno
confusion_matrix(objetivo_prueba, prediccion_logistica), confusion_matrix(objetivo_prueba, prediccion_bayes)

## mejor resultado

In [ ]:
if busqueda_logistica.best_score_ >= busqueda_bayes.best_score_:
    nombre_mejor = 'regresion logistica'
    parametros_mejor = busqueda_logistica.best_params_
    f1_prueba_mejor = f1_logistica
else:
    nombre_mejor = 'naive bayes'
    parametros_mejor = busqueda_bayes.best_params_
    f1_prueba_mejor = f1_bayes

nombre_mejor, parametros_mejor, round(f1_prueba_mejor, 4)

## conclusión

- como hay menos spam usé F1 para escoger y no solamente accuracy
- precision me dice cuántos de los marcados sí eran spam y recall cuántos spam encontró
- las matrices ayudan a ver cuáles se confundieron
- esto funciona para estos SMS en inglés, con otros mensajes tendría que volver a probar

## fuentes

- [SMS Spam Collection de UCI](https://doi.org/10.24432/C5CC84)
- [descarga de los datos](https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip)
- [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [métricas de clasificación](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics)